# Superposition in Neural Networks
## Why concepts are not neatly stored one-per-neuron

This notebook is a deep dive into one of the most important ideas for understanding why neural networks are powerful, why they are hard to interpret, and why tools like sparse autoencoders became such a big deal.


## Why I picked this

You’ve already been thinking in exactly the right direction:

- activations as points in a representation space
- neurons as axes
- whether a neuron corresponds to a feature
- whether truth, concepts, or reasoning can be localized
- how interpretability can recover meaningful structure

Superposition is where all those questions collide.


## The big idea

A naive first picture of a neural network is:

- neuron 1 = detects feature A
- neuron 2 = detects feature B
- neuron 3 = detects feature C

That picture is sometimes partly true, but in general it is **too simple**.

In real neural networks, many concepts are often **compressed into overlapping directions** in activation space rather than assigned cleanly to separate neurons.

That overlapping storage is called **superposition**.

So instead of:

- one neuron = one concept

we often get:

- one concept = a pattern across many neurons
- one neuron = mixture of many concepts

That is a major reason interpretability is hard.


## First intuition: a storage problem

Imagine you need to represent 100 meaningful features, but your layer only has 20 neurons.

If you insist on:

- one feature per neuron

you can only store 20 clean independent features.

But what if many features are rarely active at the same time?

Then the network can be clever.

It can “pack” multiple features into overlapping directions in the same 20-dimensional space, provided they usually do not interfere too badly.

That packing is efficient.
It lets the model represent more useful structure than the raw neuron count might suggest.

So superposition is not necessarily a bug.
It can be an **efficient solution to limited dimensionality**.


## Geometric picture

Suppose a layer has 2 neurons.

Then every activation is a point in a 2D plane.

If the network used neurons as clean feature slots, maybe:

- x-axis = “contains a cat”
- y-axis = “contains a car”

Nice and simple.

But the network is not forced to do that.

It could instead represent features along arbitrary directions:

$$
\text{feature A} = \begin{bmatrix}1 \\ 0\end{bmatrix},
\qquad
\text{feature B} = \begin{bmatrix}0 \\ 1\end{bmatrix},
\qquad
\text{feature C} = \begin{bmatrix}1 \\ 1\end{bmatrix},
\qquad
\text{feature D} = \begin{bmatrix}1 \\ -1\end{bmatrix}
$$

Now 4 features are living in a 2D space.

Not perfectly independently.
Not without interference.
But they are still representable.

This is the essence of superposition:
**more features than basis directions, stored through overlap**.


## Why this happens

There are two deep reasons.

### 1. The model wants efficiency
A network has finite width.
It cannot always dedicate a separate neuron to each useful pattern.

So it learns compressed distributed codes.

### 2. Many features are sparse
A lot of meaningful features are only relevant occasionally.

For example, in language:

- “is about Australian banking regulation”
- “mentions a year”
- “is sarcastic”
- “contains a city name”
- “is a question”
- “is talking about physics”
- “contains a person’s name”

Most of these are not active all the time.
Because they are sparse, the network can often reuse representational dimensions.

That is like several people sharing the same meeting room at different times.


## Simple analogy: radio frequencies in one wire

Think of one wire carrying several radio signals at once.

The wire is one physical medium, but multiple signals are mixed into it.

To a raw observer, it looks like a messy waveform.

But with the right decoder, the individual signals can be separated.

That is similar to superposition:

- activation vector = mixed waveform
- hidden features = underlying signals
- interpretability = trying to unmix them


## A concrete toy example

Let’s say we have 3 features:

- $f_1$: “animal”
- $f_2$: “vehicle”
- $f_3$: “question”

But only 2 neurons.

Let the representation vectors be:

$$
v_1 = \begin{bmatrix}1 \\ 0\end{bmatrix}, \quad
v_2 = \begin{bmatrix}0 \\ 1\end{bmatrix}, \quad
v_3 = \begin{bmatrix}1 \\ 1\end{bmatrix}
$$

If an input contains:

- animal only, activation could be $v_1$
- vehicle only, activation could be $v_2$
- question only, activation could be $v_3$

If input contains animal + question, activation might look like:

$$
v_1 + v_3 = \begin{bmatrix}2 \\ 1\end{bmatrix}
$$

Now notice what happened.

Neuron 1 no longer means a single thing.
Its value is influenced by both “animal” and “question”.

Neuron 2 is also mixed.

So looking at a single neuron is misleading.
The meaningful object is the **direction in the space**, not the individual coordinate alone.

That is why neurons are often not the right unit of interpretation.


## Why this matters for your earlier question: “Can each axis be a predefined feature?”

This is exactly the reason the answer is usually **no**.

A neuron axis is just the coordinate system we humans use to inspect that layer.
But the network’s meaningful features can live in arbitrary directions, not aligned to axes.

A concept may be:

- partly in neuron 7
- partly in neuron 18
- partly in neuron 203

And neuron 18 may also participate in several other concepts.

So the coordinate axes are often not privileged concept axes.

This is crucial.

The network does not care about our desire for neat human-readable axes.
It cares about minimizing loss.


## Basis vs features

This distinction is essential.

### Basis
A basis is just a set of coordinates used to describe vectors.

For a hidden layer, neurons give you a basis.

### Features
Features are meaningful directions/patterns in that space.

These do **not** have to align with the basis.

So:

- neurons give coordinates
- concepts may live in other directions
- interpretability often tries to find a better basis

This is one way to understand sparse autoencoders:
they try to discover a feature basis that is more interpretable than the raw neuron basis.


## Why superposition makes interpretability hard

Suppose neuron 42 lights up strongly.

What does that mean?

In a clean one-neuron-one-feature world, easy:
it means one specific feature is present.

But in superposition, neuron 42 might be participating in:

- past tense verbs
- legal terminology
- hedging language
- some sentiment pattern
- some structural syntax cue

depending on context.

So neuron activations are often **polysemantic**.

### Polysemantic neuron
A neuron that responds to multiple unrelated or weakly related features.

This is one of the classic signs of superposition.
Not always.
But often.


## Why does polysemanticity emerge?

Because the model is happy to reuse the same neuron if the different features:

- rarely co-occur
- can be separated downstream
- do not create too much destructive interference

This is similar to reusing the same memory slot for multiple things that rarely need to be accessed at the same time.

The optimization process will tolerate some overlap if it improves overall efficiency.


## The mathematical picture

Let’s formalize a little.

Suppose a hidden representation is $x \in \mathbb{R}^d$.

Now imagine there are conceptual features $f_1, f_2, \dots, f_k$, with $k > d$.

Each feature has a direction $v_i \in \mathbb{R}^d$.

Then a hidden representation can be modeled as:

$$
x = \sum_{i=1}^{k} a_i v_i
$$

where:

- $a_i$ = strength of feature $i$
- $v_i$ = direction associated with feature $i$

This is the core picture.

Now if $k > d$, then by definition you have more features than dimensions.
So multiple features must overlap.
They cannot all be orthogonal.

That overlap is superposition.


## What causes interference?

If two feature vectors $v_i$ and $v_j$ are not orthogonal, then activating one affects the readout of the other.

Their overlap is measured by the dot product:

$$
v_i^\top v_j
$$

- If this is $0$, they are orthogonal: no linear interference.
- If large, they overlap strongly.

So superposition is basically about **non-orthogonal feature packing**.

The network tries to arrange feature geometry so useful things can coexist with manageable interference.


## Why sparsity helps

This is the really beautiful part.

If features are sparse, then at any given input only a few $a_i$'s are nonzero or large.

So even if many features share the same space, only a few are active at once.
That keeps interference limited.

This is why sparse features and superposition go together so naturally.

You can think of it like this:

- dense simultaneous activation -> too much collision
- sparse activation -> shared space becomes workable

That is one reason sparse autoencoders are so useful for interpretability:
they try to recover sparse latent features from dense superposed activations.


## Important connection to sparse autoencoders

A sparse autoencoder takes a hidden activation vector $x$ and tries to write it as:

$$
x \approx \sum_{j=1}^{m} z_j d_j
$$

where:

- $d_j$ are decoder directions
- $z_j$ are latent feature activations
- only a few $z_j$ are encouraged to be active

So instead of saying:

- raw neuron coordinates are the real features

the sparse autoencoder says:

- maybe the raw activations are mixtures
- let me find a new dictionary of directions $d_j$
- and represent each input with only a few active features

That is literally an attempt to **de-superpose** the representation.

This is why SAEs matter so much in mechanistic interpretability.


## A deeper way to say it

The raw layer basis may be efficient for computation,
but not efficient for human understanding.

An SAE tries to find a new overcomplete basis:

- often more features than neurons
- sparse activations
- more interpretable directions

So paradoxically:

- network layer: compressed mixed code
- SAE feature space: expanded sparse code

This often makes concepts easier to isolate.


## Why “more features than neurons” is not impossible

This can feel paradoxical.

“How can you have more features than dimensions?”

Because “feature” does not mean “independent axis.”

A feature is just a meaningful direction or pattern.
A space of dimension $d$ can contain infinitely many directions.

What it cannot contain is infinitely many mutually orthogonal independent axes.

So the model can encode many features, but they must overlap.

That overlap is the price.


## An analogy from language

Imagine one sentence:

> “The old bank near the river collapsed.”

The word “bank” is ambiguous.

Its representation may involve mixed directions corresponding to:

- finance
- riverbank
- geography
- structural collapse
- linguistic context

The model’s internal code is not likely a neat box labeled “river sense of bank.”
Instead the surrounding context steers the activation pattern through a distributed geometry.

This is one reason language models are rich but hard to interpret.


## Superposition is not random chaos

This is important.

Superposition does **not** mean the network stores concepts in a hopeless mess.

The mixing is usually structured enough that downstream layers can still use it.

That means the geometry is not arbitrary noise.
It is an organized compressed code.

Interpretability research is trying to uncover that organization.


## Relation to distributed representation

These two ideas are related but not identical.

### Distributed representation
A concept is represented across many neurons.

### Superposition
Multiple concepts share overlapping representational directions.

Distributed representation says:
- one concept uses many neurons

Superposition says:
- many concepts reuse the same neurons/dimensions

In practice, both often happen together.


## Relation to disentanglement

You can think of disentanglement as the opposite tendency:

- one factor of variation
- one clean latent direction

That is a very attractive structure for humans.

But real networks trained only for task performance do not strongly optimize for human-friendly disentanglement.
They optimize for predictive success.

So unless the task or architecture encourages neat separation, superposition often appears.


## Why this matters for “truth directions”

This ties beautifully to your recent thinking.

Suppose there really is a “truth direction” in activation space.

That does **not** necessarily mean:

- there is one truth neuron
- truth is stored cleanly and only as one isolated component
- the model explicitly reasons symbolically over that direction

It may mean instead that:

- across many examples, truth-related structure has a consistent geometric bias
- this bias is recoverable as a direction
- but it may still be entangled with many other features

So superposition helps explain why:
- a meaningful direction can exist,
- yet raw neurons still look messy,
- and proving causal use is harder than merely identifying a direction.

That distinction is very important.


## A subtle but important point: the model may use a feature without making it human-separable

A network does not need to represent concepts in a way we can easily name.

It only needs internal states that support good computation.

So the fact that we cannot point to a single clean neuron for a feature does **not** mean the feature is absent.

It may exist as a distributed, superposed direction.

This is one reason interpretability is fundamentally about geometry, not just neuron inspection.


## Why downstream layers can still work with superposed features

You might ask:
“If features overlap, how can later layers use them reliably?”

Because downstream linear transformations can read out specific directions.

If the next layer has weight vector $w$, then it computes something like:

$$
w^\top x
$$

If $x$ contains several mixed features, $w$ can still be tuned to become sensitive to a particular useful direction.

So even when features are superposed, the model can often selectively read them out.

This is like having several voices in a room but training a microphone to focus on one pattern.


## The tradeoff

Superposition gives the model:

- compression
- reuse of dimensions
- higher representational efficiency

But it costs:

- interpretability
- clean modularity
- reduced human-readability
- possible interference

So it is a tradeoff between **efficiency** and **clarity**.

Optimization often prefers efficiency.
Humans prefer clarity.

That tension is central to mech interp.


## Why mechanistic interpretability cares so much

Mechanistic interpretability wants to answer things like:

- what features exist?
- where are they?
- how are they combined?
- what circuits use them?
- which computations depend on them?

If features were neatly one-per-neuron, this would be much easier.

But because of superposition, mech interp needs stronger tools:

- feature dictionaries
- sparse autoencoders
- activation patching
- causal interventions
- linear probes
- circuit analysis

Superposition is one of the reasons the field exists in its modern form.


## One clean mental model to keep

Here is the mental picture I want you to retain:

### A hidden layer is not best imagined as a list of labeled boxes.
It is better imagined as a **vector space** containing many meaningful directions.

- neurons = coordinates
- features = directions/patterns
- activations = points in that space
- superposition = multiple features overlapping in shared dimensions
- interpretability = trying to recover the hidden feature geometry

That one picture will help you enormously.


## Let me push one level deeper: why orthogonal features are special

If two features are orthogonal, then they are linearly independent and separable without interference.

But if many sparse features exist and width is limited, the model cannot afford to keep them all orthogonal.

So it arranges them in partially overlapping directions.

In some toy models, you can think of the optimization as trying to place many feature vectors in a space so that:

- commonly co-occurring important features interfere less
- rarely co-occurring features can overlap more

That means the learned geometry reflects statistics of the data.

This is profound.

It suggests internal geometry is shaped not just by abstract logic, but by:
- co-occurrence structure
- sparsity
- task demands
- layer width
- optimization constraints


## A powerful consequence

The model’s representation is not merely “storing meanings.”
It is storing them under compression pressure.

So the internal concepts we observe are not pure ideal meanings.
They are **task-shaped compressed computational objects**.

That is why a feature can look weird, mixed, or unexpectedly broad.


## Common mistake to avoid

Do not think:

> “If a concept is real, there must be one neuron for it.”

Better:

> “If a concept is real, it may correspond to some robust direction, subspace, sparse feature, or circuit — not necessarily a single neuron.”

That is a much more mature interpretability mindset.


## A concise summary

Superposition means a neural network can store many more meaningful features than it has neurons by encoding them in overlapping directions of activation space. This makes representations efficient, especially when features are sparse and rarely co-occur, but it also makes neurons polysemantic and interpretability difficult. Sparse autoencoders are powerful because they try to reverse this mixing and recover a more human-readable feature basis.


## Three questions for you to think about

Don’t answer immediately unless you want to really engage with it:

1. If a model has width $d$, why is it still possible for it to represent more than $d$ meaningful features?
2. Why does sparsity make superposition more workable?
3. Why might a “truth direction” exist even when no individual neuron corresponds cleanly to truth?

And for the next step, I’d recommend we continue directly into **how sparse autoencoders mathematically decompose superposed representations**, with equations and a toy numerical example.
